# 4. Document Q&A (RAG)
Build a small vector index over local policy files.

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

In [ ]:
docs=[]
for p in Path('../data').glob('*.txt'):
    loaded=TextLoader(str(p), encoding='utf-8').load()
    for d in loaded: d.metadata['file_name']=p.name
    docs.extend(loaded)
chunks=RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60).split_documents(docs)
store=FAISS.from_documents(chunks, OpenAIEmbeddings(model='text-embedding-3-small'))

In [ ]:
question='Which plan includes audit logs?'
retrieved=store.similarity_search(question, k=3)
context='\n\n'.join(f"SOURCE: {d.metadata['file_name']}\n{d.page_content}" for d in retrieved)
model=init_chat_model('openai:gpt-4.1-mini', temperature=0)
response=model.invoke(f'Answer only from this context and cite the source file.\n\n{context}\n\nQuestion: {question}')
response.content

## Experiment
Change chunk size and top-k. Observe precision, context completeness, and cost.